# Trajectories of magnetic soft robot

### Imports

In [1]:
import numpy as np
import jax.numpy as jnp
import jax
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from softmobility import SoftBody, FlowBodySolver, shear_flow, oscillating_magnetic_field
from softmobility.classes.flowbodysolver import _rotation_matrix_from_Rodrigues_impl
from softmobility.classes.flowbodysolver import compute_Bortz_operator, rescale_orientation

np.set_printoptions(precision=3, linewidth=100, suppress=True, sign=" ")

### Magnetic parameters

In [2]:
MAGNETIC_FORCING = 100  # Magnetic amplitude: m Bx / (mu a omega)
MAGNETIC_RATIO = 1      # Magnetic ratio: By /Bx

## Numerical simulation

### Parameter file

In [3]:
yaml_data = """
# Variable Prefixes (Optional)
dof_names:
  - phi
design_names:      
  - spring_k    
  - radius
input_names:
  - magnetic
    
# Default Values (Optional)
defaults:
  length: 0.
  _ref_: 1
  spring_k: .84            
  radius: .5
  _sti_: 100
  _rad_: 2
  _len_: 2

# Spheres (Mandatory)
spheres:
  - # Sphere 0 #################
    radius: _ref_                    
    orientation: [0, 0, phi]
    torque: 
      - 0
      - 0
      - "magnetic1 * cos(phi) - magnetic0 * sin(phi) - _sti_ * spring_k * phi"              

  - # Sphere 1 #################
    radius: radius               
    position: [_ref_ + _rad_ * radius + _len_ * length, 0, 0]       
    torque: [0, 0, _sti_ * spring_k * phi]    
"""

### Soft Body

In [4]:
mybody= SoftBody(yaml_data, verbose=True)

  Found variables: phi, radius, spring_k, magnetic0, magnetic1, magnetic2
  3D field inputs:  ['magnetic']
    Sphere 0
      Radius: 1
      Position: ['0', '0', '0']
      Orientation: ['0', '0', 'phi']
      Coupling matrix C_H:
[['0', '0', '0'], ['0', '0', '0'], ['0', '0', '0'], ['0', '0', '0'], ['0', '0', '0'], ['-sin(phi)', 'cos(phi)', '0']]
      Coupling matrix C_K:
[['0'], ['0'], ['0'], ['0'], ['0'], ['-100*spring_k']]
    Sphere 1
      Radius: radius
      Position: ['2*radius + 1', '0', '0']
      Orientation: ['0', '0', '0']
      Coupling matrix C_H:
[['0', '0', '0'], ['0', '0', '0'], ['0', '0', '0'], ['0', '0', '0'], ['0', '0', '0'], ['0', '0', '0']]
      Coupling matrix C_K:
[['0'], ['0'], ['0'], ['0'], ['0'], ['100*spring_k']]


### Flow-body solver

In [5]:
# Create a shear flow with shear rate 1
no_flow = shear_flow(shear_rate=0)
# Create a magnetic field 
MAGNETIC_FIELD = oscillating_magnetic_field(amp_x=MAGNETIC_FORCING, amp_y=MAGNETIC_RATIO * MAGNETIC_FORCING, omega=1)

print(MAGNETIC_FIELD.vector(time=0))
print(MAGNETIC_FIELD.vector(time=0.2))

[ 100.    0.    0.]
[ 100.      19.867    0.   ]


In [6]:
# parameters
final_time = 12 * 2 * jnp.pi  
DT = 0.1

# optimal design parameters
# mybody.set_design_defaults(new_dict={'spring_k': .172, 'radius': .387})

solver = FlowBodySolver(
    soft_body=mybody, 
    flow=no_flow, 
    input_map={"magnetic": MAGNETIC_FIELD}, 
    dt=DT, 
    init_orientation=[0, 0, 0], 
    integrator="RK2")
trajectory = solver.simulate(final_time=final_time)

Time: 0.000 / 75.398  Integrator RK2
Time: 10.000 / 75.398  Integrator RK2
Time: 20.000 / 75.398  Integrator RK2
Time: 30.000 / 75.398  Integrator RK2
Time: 40.000 / 75.398  Integrator RK2
Time: 50.000 / 75.398  Integrator RK2
Time: 60.000 / 75.398  Integrator RK2
Time: 70.000 / 75.398  Integrator RK2


### Figure of trajectory

In [7]:
# Extract position, orientation and dofs
positions = jnp.array([t[0] for t in trajectory])
orientations = jnp.array([t[1] for t in trajectory])
dofs = jnp.array([t[2][0] for t in trajectory])

# Time vector
time = jnp.linspace(0, final_time, len(trajectory))

# Create a subplot figure with 3 rows and 1 column
fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=("DOF", "Position (X, Y, Z)", "Orientation (X, Y, Z)"),
    shared_xaxes=True,  # Share the x-axis for all plots (time)
    vertical_spacing=0.1  # Adjust vertical spacing (reduce clutter)
)

# Plot DOFs in the first subplot
fig.add_trace(go.Scatter(x=time, y=dofs[:], mode='lines', name='DOF 0'), row=1, col=1)

# Plot position components (X, Y, Z) in the second subplot
fig.add_trace(go.Scatter(x=time, y=positions[:, 0], mode='lines', name='Position X'), row=2, col=1)
fig.add_trace(go.Scatter(x=time, y=positions[:, 1], mode='lines', name='Position Y'), row=2, col=1)
fig.add_trace(go.Scatter(x=time, y=positions[:, 2], mode='lines', name='Position Z'), row=2, col=1)

# Plot orientation components (X, Y, Z) in the third subplot
fig.add_trace(go.Scatter(x=time, y=orientations[:, 0], mode='lines', name='Orientation X'), row=3, col=1)
fig.add_trace(go.Scatter(x=time, y=orientations[:, 1], mode='lines', name='Orientation Y'), row=3, col=1)
fig.add_trace(go.Scatter(x=time, y=orientations[:, 2], mode='lines', name='Orientation Z'), row=3, col=1)

# Update layout for titles and labels
fig.update_layout(
    title="Trajectory Data Over Time",
    xaxis_title="Time (t)",
    # yaxis_title="Values (Position/Orientation Components)",
    template="plotly_white",  # Set the background to white
    showlegend=True,  # Show legend to distinguish between the traces
    height=800,  # Set figure height
    width=800,   # Set figure width
    plot_bgcolor='white',  # Set the plot background to white
    paper_bgcolor='white'  # Set the paper background to white
)

# Show the plot
fig.show()

## Numerical optimization

### Parameters

In [8]:
N_DT = 64               # number of time steps per period
N_TRANSIENT = 4         # number of periods before optimization begins
MAGNETIC_FORCING = 100  # Magnetic amplitude: m Bx / (mu a omega)
MAGNETIC_RATIO = 1      # Magnetic ratio: By /Bx

DT = 2.0 * np.pi / N_DT
MAGNETIC_FIELD = oscillating_magnetic_field(amp_x=MAGNETIC_FORCING, amp_y=MAGNETIC_RATIO * MAGNETIC_FORCING, omega=1)

### Functions

In [9]:
rotation_matrix_from_Rodrigues = jax.jit(
    lambda rvec, Ndof: _rotation_matrix_from_Rodrigues_impl(rvec, Ndof), static_argnums=(1,)
)

In [10]:
def sixc_velocity(design, orientation, dofs, field_lab):
    rot_matrix, sixc_rot_matrix = rotation_matrix_from_Rodrigues(orientation, Ndof=mybody.Ndof)
    field_body = rot_matrix.T @ field_lab
    tensors = mybody.compute_mobility_problem(dofs, design)
    sixc_velocity = tensors.M_H @ field_body + tensors.M_K @ dofs

    return sixc_rot_matrix @ sixc_velocity 

In [11]:
def _rollout_one_period(design, position, orientation, dofs):

    def step(carry, t):
        position, orientation, dofs = carry
        time = t * DT
        current_position = position.copy()
        current_orientation = orientation.copy()
        current_dofs = dofs.copy()

        field_lab = MAGNETIC_FIELD.vector(position, time)
        bortz     = compute_Bortz_operator(orientation)
        p1        = sixc_velocity(design, orientation, dofs, field_lab)
        
        k1_pos = p1[:3]
        k1_ori = p1[3:6]
        k1_dof = p1[6:]

        position    = position    + DT * k1_pos / 2
        orientation = orientation + DT * bortz @ k1_ori / 2
        dofs        = dofs        + DT * k1_dof / 2
        
        field_lab = MAGNETIC_FIELD.vector(position, time)
        p2        = sixc_velocity(design, orientation, dofs, field_lab)  

        k2_pos = p2[:3]
        k2_ori = p2[3:6]
        k2_dof = p2[6:]

        position    = current_position    + DT * (k1_pos + k2_pos) / 2
        orientation = current_orientation + DT * bortz @ (k1_ori + k2_ori) / 2
        orientation = rescale_orientation(orientation)
        dofs        = current_dofs        + DT * (k1_dof + k2_dof) / 2
                
        return (position, orientation, dofs), None

    timesteps = jnp.arange(N_TRANSIENT * N_DT)
    (position, orientation, dofs), _ = jax.lax.scan(
        step, (position, orientation, dofs), timesteps
    )

    x_init    = position[0].copy()   # capture before evaluation
    timesteps = jnp.arange(N_DT)
    (position, orientation, dofs), _ = jax.lax.scan(
        step, (position, orientation, dofs), timesteps
    )
    mean_vx = (position[0] - x_init) / (2. * np.pi)   # displacement from true start

    return position, orientation, dofs, mean_vx

rollout_one_period = jax.jit(_rollout_one_period)

In [12]:
def _mean_vx(design, position, orientation, dofs):
    return rollout_one_period(design, position, orientation, dofs)[-1]

grad_vx_mean = jax.jit(jax.grad(_mean_vx, argnums=0))

In [13]:
def optimize_online(init_design, n_warmup=10, n_periods=200, learning_rate=1e-3, momentum=0.9, maximize=True):
    """
    Gradient ascent on design over multiple periods, with warmup transient.
    """
    design      = jnp.array(init_design)
    position    = jnp.array([0., 0., 0.])
    orientation = jnp.array([0., 0., 0.])
    dofs        = jnp.zeros(mybody.Ndof)
    velocity    = jnp.zeros_like(design)
    if maximize:
        sign = +1
    else:
        sign = -1

    # --- Warmup: run n_warmup periods without gradient update ---
    print("Warming up...")
    for _ in range(n_warmup):
        position, orientation, dofs, mean_vx = rollout_one_period(design, position, orientation, dofs)
        
    print(f"Warmup done — position={position[0]}, mean_vx={float(mean_vx):.6f}")

    # --- Gradient ascent ---
    for period in range(n_periods):
        grad = sign * grad_vx_mean(design, position, orientation, dofs)
                
        if jnp.any(jnp.isnan(grad)):
            print(f"  Warning: NaN gradient at period {period}, skipping update")
        else:
            velocity = momentum * velocity + learning_rate * grad / (jnp.linalg.norm(grad) + 1.E-6)
            design   = jnp.clip(design + velocity, 3 * learning_rate, 1.0)

        # Advance state with updated design as buffer
        position, orientation, dofs, mean_vx = rollout_one_period(design, position, orientation, dofs)
            
        if period % 100 == 0:
            print(f"period {period:4d}  mean_vx={float(mean_vx):.6f}  "
                  f"|grad|={jnp.linalg.norm(grad):.4f}  design={design}")

    return design

### Simulation and optimization

In [14]:
opt_design = optimize_online(
    init_design=mybody.design_defaults,
    n_warmup=N_TRANSIENT,
    n_periods=1001,
    learning_rate=0.001,
    momentum=0.8,
    maximize=True,
)

print("\nOPTIMUM DESIGN PARAMETERS:\n", mybody.design_variables)
print(opt_design)

Warming up...
Warmup done — position=0.43863046169281006, mean_vx=0.003021
period    0  mean_vx=0.003029  |grad|=0.0085  design=[ 0.501  0.839]
period  100  mean_vx=0.005920  |grad|=0.0054  design=[ 0.526  0.396]
period  200  mean_vx=0.006653  |grad|=0.0002  design=[ 0.415  0.215]
period  300  mean_vx=0.006652  |grad|=0.0002  design=[ 0.415  0.215]
period  400  mean_vx=0.006652  |grad|=0.0002  design=[ 0.415  0.215]
period  500  mean_vx=0.006652  |grad|=0.0002  design=[ 0.415  0.215]
period  600  mean_vx=0.006652  |grad|=0.0002  design=[ 0.415  0.215]
period  700  mean_vx=0.006674  |grad|=0.0002  design=[ 0.415  0.215]
period  800  mean_vx=0.006674  |grad|=0.0002  design=[ 0.415  0.215]
period  900  mean_vx=0.006674  |grad|=0.0002  design=[ 0.415  0.215]
period 1000  mean_vx=0.006674  |grad|=0.0002  design=[ 0.415  0.215]

OPTIMUM DESIGN PARAMETERS:
 ['radius', 'spring_k']
[ 0.415  0.215]


## Theoretical approach, limit of small angles

### Parameter file

In [15]:
yaml_data = """
# Variable Prefixes (Optional)
dof_names:
  - phi
design_names:      
  - stiffness    
  - radius
  - length
input_names:
  - magnetic
    
# Default Values (Optional)
defaults:
  stiffness: 0.1            
  radius: 0.25
  length: 0.5
  stiffness_factor: 100
  radius_factor: 2
  length_factor: 2
  moment: 1
  refradius: 1
  
# Spheres (Mandatory)
spheres:
  - # Sphere 0 #################
    radius: refradius                    
    orientation: [0, 0, phi]
    torque: 
      - 0
      - 0
      - "moment * magnetic - stiffness_factor * stiffness * phi"              

  - # Sphere 1 #################
    radius: radius_factor * radius               
    position: [refradius + radius_factor * radius + length_factor * length, 0, 0]       
    torque: [0, 0, stiffness_factor * stiffness * phi]    
"""

### Extracting mobility components

In [16]:
theory_body= SoftBody(yaml_data, verbose=True)

  Found variables: phi, length, radius, stiffness, magnetic
  Scalar inputs:    ['magnetic']
    Sphere 0
      Radius: 1
      Position: ['0', '0', '0']
      Orientation: ['0', '0', 'phi']
      Coupling matrix C_H:
[['0'], ['0'], ['0'], ['0'], ['0'], ['1']]
      Coupling matrix C_K:
[['0'], ['0'], ['0'], ['0'], ['0'], ['-100*stiffness']]
    Sphere 1
      Radius: 2*radius
      Position: ['2*length + 2*radius + 1', '0', '0']
      Orientation: ['0', '0', '0']
      Coupling matrix C_H:
[['0'], ['0'], ['0'], ['0'], ['0'], ['0']]
      Coupling matrix C_K:
[['0'], ['0'], ['0'], ['0'], ['0'], ['100*stiffness']]


In [17]:
tensors = theory_body.compute_mobility_problem()
print("\nMobility Matrix M_H (multiplied by mu):")
print(tensors.M_H)
print("\nMobility Matrix M_K (multiplied by mu):")
print(tensors.M_K)


Mobility Matrix M_H (multiplied by mu):
[[ 0.   ]
 [ 0.   ]
 [ 0.   ]
 [ 0.   ]
 [ 0.   ]
 [ 0.002]
 [ 0.037]]

Mobility Matrix M_K (multiplied by mu):
[[ 0.   ]
 [-0.141]
 [ 0.   ]
 [ 0.   ]
 [ 0.   ]
 [ 0.168]
 [-0.542]]


In [18]:
m1 = tensors.M_H[1,0]
m2 = tensors.M_H[-2,0]
m3 = tensors.M_H[-1,0]

k1 = tensors.M_K[1,0] 
k2 = tensors.M_K[-2,0]
k3 = tensors.M_K[-1,0]

print("Magnetic coupling coefficients:", m1, m2, m3)
print("Elastic  coupling coefficients:", k1, k2, k3) 

Magnetic coupling coefficients: 9.2426315e-05 0.0023460574 0.03739889
Elastic  coupling coefficients: -0.14099357 0.1676875 -0.5416764


In the limit of small angles we have
$$ \dot \theta_0 = -m_2 F (\theta_0 + \phi) + m_2 F R \sin(\omega t) + k_2 \phi, $$
$$ \dot \phi     = -m_3 F (\theta_0 + \phi) + m_3 F R \sin(\omega t) + k_3 \phi, $$
with $F = m B_x /(\mu a \omega)$, the amplitude of the magnetic forcing (along the steady direction), and $R = B_y/ B_x$ the ratio of magnetic components between the oscillating and steady ones. 

Calculating mean velocity in this limit with Wolfram calculation `Magnetic soft robot oneDOF`.

In [19]:
vmoy = (MAGNETIC_FORCING**2*(k2*m1 - k1*m2)*m3)/(2*((1 + k3**2)*(1 + MAGNETIC_FORCING**2*m2**2) - 2*MAGNETIC_FORCING*(k2 + k3 + MAGNETIC_FORCING*(-1 + k2*k3)*m2)*m3 + MAGNETIC_FORCING**2*(1 + k2**2)*m3**2))

print("Mean velocity normalized by R**2:", vmoy)
print("Distance for 5 periods:", vmoy * 10 * np.pi * 0.1**2)

Mean velocity normalized by R**2: 0.0031653966
Distance for 5 periods: 0.0009944388


### Optimization for given magnetic forcing

In [20]:
def optimize_velocity(
    init_design,
    learning_rate=0.03,
    max_iters=1000,
    momentum = 0.9,  # Tune this (0.8 - 0.95 works well)
    forcing = MAGNETIC_FORCING
):
    
    @jax.jit
    def mean_v(design):
        dofs = jnp.zeros(theory_body.Ndof)
        tensors = theory_body.compute_mobility_problem(dofs, design)

        m1 = tensors.M_H[1,0]
        m2 = tensors.M_H[-2,0]
        m3 = tensors.M_H[-1,0]

        k1 = tensors.M_K[1,0]
        k2 = tensors.M_K[-2,0]
        k3 = tensors.M_K[-1,0]
        
        vmean = (forcing**2*(k2*m1 - k1*m2)*m3)/(2*((1 + k3**2)*(1 + forcing**2*m2**2) - 2*forcing*(k2 + k3 + forcing*(-1 + k2*k3)*m2)*m3 + forcing**2*(1 + k2**2)*m3**2))

        return vmean

    def constraints_design(design):
        return jnp.clip(design, 0, 1)

    design = init_design
    grad_mean_v = jax.jit(jax.grad(mean_v))
    velocity = jnp.zeros_like(design) 
        
    for i in range(max_iters):
        grad = grad_mean_v(design)
        grad_norm = jnp.linalg.norm(grad)

        if i % 100 == 0:
            print(i, mean_v(design), design, grad_norm)

        # Update velocity (exponential moving average of gradients)
        velocity = momentum * velocity + learning_rate * grad / (grad_norm + 1E-6)
        design = design + velocity

        # Enforce any additional constraints (like bounding design between 0 and 1)
        design = constraints_design(design)

    return design, mean_v(design)    

In [21]:
# Optimization trial
init_params = theory_body.design_defaults
opt_design, value = optimize_velocity(
    init_design=init_params,
    learning_rate=1E-3, 
    max_iters=2001,
    forcing=100,
)

0 0.0031653966 [ 0.5   0.25  0.1 ] 0.02730152
100 0.01596637 [ 0.     0.328  0.353] 0.024827765
200 0.015993915 [ 0.     0.334  0.381] 0.025289837
300 0.015993943 [ 0.     0.334  0.382] 0.025320549
400 0.015993945 [ 0.     0.334  0.382] 0.025321886
500 0.015993945 [ 0.     0.334  0.382] 0.025321912
600 0.015993945 [ 0.     0.334  0.382] 0.025321912
700 0.015993945 [ 0.     0.334  0.382] 0.025321912
800 0.015993945 [ 0.     0.334  0.382] 0.025321912
900 0.015993945 [ 0.     0.334  0.382] 0.025321912
1000 0.015993945 [ 0.     0.334  0.382] 0.025321912
1100 0.015993945 [ 0.     0.334  0.382] 0.025321912
1200 0.015993945 [ 0.     0.334  0.382] 0.025321912
1300 0.015993945 [ 0.     0.334  0.382] 0.025321912
1400 0.015993945 [ 0.     0.334  0.382] 0.025321912
1500 0.015993945 [ 0.     0.334  0.382] 0.025321912
1600 0.015993945 [ 0.     0.334  0.382] 0.025321912
1700 0.015993945 [ 0.     0.334  0.382] 0.025321912
1800 0.015993945 [ 0.     0.334  0.382] 0.025321912
1900 0.015993945 [ 0.     0.

In [22]:
print(theory_body.design_variables)
print(opt_design * np.array([2, 2, 100]))

['length', 'radius', 'stiffness']
[  0.      0.669  38.247]


### Optimization for different magnetic forcing

In [30]:
forcings = 10**np.linspace(start=4, stop=-1, num=20,endpoint=True)
opt_length = np.zeros_like(forcings)
opt_radius = np.zeros_like(forcings)
opt_stiffness = np.zeros_like(forcings)
opt_speed = np.zeros_like(forcings)

init_design = opt_design
for i, f in enumerate(forcings):
    if i==0:
        iters = 1001
    else:
        iters = 201
    init_design, value = optimize_velocity(
        init_design=init_design,
        learning_rate=1E-3, 
        max_iters=iters,
        forcing=f,
    )
    print(i, f, init_design)
    opt_length[i] = init_design[0] * 2
    opt_radius[i] = init_design[1] * 2
    opt_stiffness[i] = init_design[2] * 100
    opt_speed[i] = value

0 0.026713584 [ 0.     0.519  0.857] 0.042118043
100 0.02671478 [ 0.     0.517  0.854] 0.041905962
200 0.026715096 [ 0.     0.516  0.851] 0.041910853
300 0.026715288 [ 0.     0.515  0.849] 0.041913703
400 0.026715409 [ 0.     0.515  0.847] 0.04191561
500 0.026715461 [ 0.     0.515  0.846] 0.04191699
600 0.026715493 [ 0.     0.514  0.845] 0.041918095
700 0.026715478 [ 0.     0.514  0.844] 0.04191885
800 0.026715487 [ 0.     0.514  0.844] 0.04191938
900 0.026715511 [ 0.     0.514  0.844] 0.04191984
1000 0.026715534 [ 0.     0.514  0.843] 0.041920118
0 10000.0 [ 0.     0.514  0.843]
0 0.026555965 [ 0.     0.514  0.843] 0.04191404
100 0.026556823 [ 0.     0.512  0.841] 0.04174084
200 0.026557118 [ 0.     0.511  0.838] 0.04174454
1 5455.59478116852 [ 0.     0.511  0.838]
0 0.026267994 [ 0.     0.511  0.838] 0.041736748
100 0.026271645 [ 0.     0.507  0.831] 0.041421343
200 0.026273103 [ 0.     0.506  0.826] 0.0414287
2 2976.351441631319 [ 0.     0.506  0.826]
0 0.02575544 [ 0.     0.506  0.

In [29]:
fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.1,
)

# Subplot 1: length and radius
fig.add_trace(go.Scatter(x=forcings, y=opt_length,    mode='lines+markers', name='Length'), row=1, col=1)
fig.add_trace(go.Scatter(x=forcings, y=opt_radius,    mode='lines+markers', name='Radius'), row=1, col=1)

# Subplot 2: stiffness
fig.add_trace(go.Scatter(x=forcings, y=opt_stiffness, mode='lines+markers', name='Stiffness'), row=2, col=1)

# Subplot 3: speed
fig.add_trace(go.Scatter(x=forcings, y=opt_speed, mode='lines+markers', name='Speed'), row=3, col=1)


axis_style = dict(
    showline=True, linewidth=1, linecolor="black", mirror=True,
    showgrid=False, zeroline=False,
    ticks="inside", tickwidth=1, tickcolor="black",
)

fig.update_layout(
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(family="serif", size=12, color="black"),
    height=1000, width=600,
    legend=dict(bordercolor="black", borderwidth=1, bgcolor="white"),
    xaxis= dict(**axis_style, showticklabels=False, type="log"),
    xaxis2= dict(**axis_style, showticklabels=False, type="log"),
    xaxis3=dict(**axis_style, title="Magnetic Forcing",      type="log"),
    yaxis= dict(**axis_style, title="Length, Radius"),
    yaxis2=dict(**axis_style, title="Stiffness"),
    yaxis3=dict(**axis_style, title="Speed", type="log")
)

fig.show()

### Limit of large magnetic fields

In [25]:
def optimize_velocity_largeB(
    init_design,
    learning_rate=0.03,
    max_iters=1000,
    momentum = 0.9,  # Tune this (0.8 - 0.95 works well)
):
    def mean_v(design):
        dofs = jnp.zeros(theory_body.Ndof)
        tensors = theory_body.compute_mobility_problem(dofs, design)

        m1 = tensors.M_H[1,0]
        m2 = tensors.M_H[-2,0]
        m3 = tensors.M_H[-1,0]

        k1 = tensors.M_K[1,0]
        k2 = tensors.M_K[-2,0]
        k3 = tensors.M_K[-1,0]
        
        vmean = ((k2*m1 - k1*m2)*m3)/(2*((1 + k3**2)*m2**2 - 2*(-1 + k2*k3)*m2*m3 + (1 + k2**2)*m3**2))

        return vmean

    def constraints_design(design):
        return jnp.clip(design, 0, 1)

    design = init_design
    grad_mean_v = jax.jit(jax.grad(mean_v))
    velocity = jnp.zeros_like(design) 
        
    for i in range(max_iters):
        grad = grad_mean_v(design)
        grad_norm = jnp.linalg.norm(grad)

        if i % 100 == 0:
            print(i, mean_v(design), design, grad_norm)

        # Update velocity (exponential moving average of gradients)
        velocity = momentum * velocity + learning_rate * grad / (grad_norm + 1E-6)
        design = design + velocity

        # Enforce any additional constraints (like bounding design between 0 and 1)
        design = constraints_design(design)

    return design, mean_v(design)    

In [26]:
# Optimization trial
init_params = theory_body.design_defaults
opt_design, value = optimize_velocity_largeB(
    init_design=init_params,
    learning_rate=1E-3, 
    max_iters=2001,
)

0 0.0039566304 [ 0.5   0.25  0.1 ] 0.03949316
100 0.025630837 [ 0.     0.427  0.506] 0.041510284
200 0.026571408 [ 0.     0.465  0.653] 0.041504838
300 0.026773171 [ 0.     0.484  0.721] 0.041946936
400 0.026846182 [ 0.     0.495  0.762] 0.042086594
500 0.026877826 [ 0.     0.502  0.789] 0.042134497
600 0.026892822 [ 0.     0.507  0.808] 0.04214918
700 0.026900424 [ 0.     0.51   0.821] 0.042151164
800 0.026904343 [ 0.     0.512  0.83 ] 0.042148564
900 0.026906442 [ 0.     0.514  0.837] 0.042144474
1000 0.026907591 [ 0.     0.515  0.842] 0.042140413
1100 0.02690823 [ 0.     0.516  0.846] 0.042136773
1200 0.026908565 [ 0.     0.517  0.849] 0.04213376
1300 0.026908766 [ 0.     0.518  0.851] 0.04213131
1400 0.026908867 [ 0.     0.518  0.853] 0.04212931
1500 0.026908945 [ 0.     0.518  0.854] 0.0421279
1600 0.026908997 [ 0.     0.519  0.855] 0.042126674
1700 0.026908996 [ 0.     0.519  0.855] 0.042125754
1800 0.026909016 [ 0.     0.519  0.856] 0.04212512
1900 0.02690904 [ 0.     0.519  0.8

In [27]:
print(theory_body.design_variables)
print(opt_design * np.array([2, 2, 100]))

['length', 'radius', 'stiffness']
[  0.      1.038  85.651]
